# **ENGINE WAKTU APLIKASI PUPUK 2025**

# Library and Utilities

In [26]:
import pytz
import sys
import pandas as pd
import gspread
from datetime import datetime, timedelta

from oauth2client.service_account import ServiceAccountCredentials
# from google.colab import drive
# drive.mount('/content/gdrive')

In [27]:
import tkinter as tk
from tkinter import ttk  # For modern widgets (optional)

In [28]:
#Autentikasi
json_path = "D:/Dump/Python/fourth-landing-316602-e06c4c4e3ba6.json"
sheet_url = "https://docs.google.com/spreadsheets/d/1f9taqCGKFtFVDmNIWFgqujf8yJLmshCFJ7j4_CgGl2Q/edit?usp=sharing"
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

In [29]:
# Grup Pupuk
fertilizer_groups = {
    "NPK": ["NPK 13", "NPK 15", "NPK 12"],
    "Dolomite": ["Dolomite"],
    "Urea": ["Urea"],
    "MOP": ["MOP"],
    "HGFB": ["HGFB"],
    "CuSO4": ["CuSO4"],
    "Zincop": ["Zincop Chelated"],
    "Kieserite": ["Kieserite"],
    "RP": ["RP"],
    "Kaptan": ["Kaptan"],
    "TSP": ["TSP"]
}

synergize_groups = {
    "NPK": ["Urea", "Kieserite", "MOP"],
    "Urea": ["NPK", "Kieserite", "MOP"],
    "RP": ["Kieserite", "Dolomite"],
    "Kieserite": ["NPK", "Urea", "RP"],
    "Dolomite": ["Kaptan", "RP"],
    "MOP": ["NPK", "Urea"],
    "HGFB": ["Zincop Chelated", "CuSO4"],
    "Zincop": ["HGFB", "CuSO4"],
    "CuSO4": ["HGFB", "Zincop"],
    "Kaptan": ["Dolomite"],
}

super_slow = {
    "Dolomite": ["Dolomite"]
}

hygroscopic = {
    "Urea": ["Urea"],
    "HGFB": ["HGFB"],
    "CuSO4": ["CuSO4"],
    "MOP": ["MOP"]
}

estate = ["Inti", "Plasma"]

interval_table = {
    "NPK": {"NPK": 60, "Urea": 14, "RP": 30, "TSP": 30, "Kieserite": 14, "Dolomite": 30, "MOP": 14, "HGFB": 30, "Zincop": 30},
    "Urea": {"NPK": 14, "Urea": 60, "RP": 30, "TSP": 30, "Kieserite": 14, "Dolomite": 30, "MOP": 14, "HGFB": 30, "Zincop": 30},
    "RP": {"NPK": 30, "Urea": 30, "RP": 60, "TSP": 60, "Kieserite": 14, "Dolomite": 14, "MOP": 30, "HGFB": 30, "Zincop": 30},
    "TSP": {"NPK": 30, "Urea": 30, "RP": None, "TSP": 30, "Kieserite": 30, "Dolomite": 30, "MOP": 30, "HGFB": 30, "Zincop": 30},
    "Kieserite": {"NPK": 14, "Urea": 14, "RP": 14, "TSP": 30, "Kieserite": 60, "Dolomite": 60, "MOP": 30, "HGFB": 14, "Zincop": 30},
    "Dolomite": {"NPK": 30, "Urea": 30, "RP": 14, "TSP": 30, "Kieserite": None, "Dolomite": 30, "MOP": 30, "HGFB": 30, "Zincop": 30},
    "MOP": {"NPK": 14, "Urea": 14, "RP": 30, "TSP": 30, "Kieserite": 30, "Dolomite": 30, "MOP": 60, "HGFB": 30, "Zincop": 30},
    "HGFB": {"NPK": 30, "Urea": 30, "RP": 30, "TSP": 30, "Kieserite": 14, "Dolomite": 30, "MOP": 30, "HGFB": 60, "Zincop": 14},
    "Zincop": {"NPK": 30, "Urea": 30, "RP": 30, "TSP": 30, "Kieserite": 30, "Dolomite": 30, "MOP": 30, "HGFB": 14, "Zincop": 60},
}

In [30]:
class FertilizerAnalysisComplete(Exception):
       pass
border_line = "=================================================================================="

# Functions

In [31]:
def get_missing_dates(df, estate_name, current_time_date):
  # Filter the DataFrame to include only records for the specified estate
  estate_data = df[(df['Estate'] == estate_name)]

  # Get the last reported time
  if not estate_data.empty:
    last_reported_time = estate_data['Date'].iloc[-1].date()
  else:
    last_reported_time = None  # Return None if no data found for the estate

  # Check for missing dates between the last reported time and the current time
  if last_reported_time:
    missing_dates = pd.date_range(last_reported_time + pd.Timedelta(days=1), current_time_date - pd.Timedelta(days=1))
    total_missing_dates = len(missing_dates)
  else:
    missing_dates = pd.DatetimeIndex([])  # Empty DatetimeIndex if no last reported time
    total_missing_dates = 0

  return missing_dates, last_reported_time, total_missing_dates

In [32]:
def format_datetime(dt):
    dt_copy = dt
    return dt_copy.strftime('%d/%m/%Y')

def format_datetimehour(dt):
    dt_copy = dt
    return dt_copy.strftime('%d/%m/%Y %H:%M:%S')

In [33]:
def validate_date(last_reported_time, current_time_date):
  if  last_reported_time == current_time_date:
    return True
  else:
    return False

In [34]:
def calculate_rainfall(in_df, current_time_date, current_daily_rainfall, estate_name):

  # Create cache data
  df = in_df.copy()

  # Filter the DataFrame to include only records for the specified estate
  df = df[(df['Estate'] == estate_name)]

  # Hitung Accumulation Rainfall -29 days
  df.loc[df.index[-1], 'Accumulation Rainfall -29 days'] = df['Daily Rainfall (mm)'].iloc[-29:].sum()
  # print("df", df)

  # Hitung Evapotranspiration (dibagi 30 sesuai logic Excel)
  evapotranspiration = (120 if len(df.iloc[-29:]) > 10 else 150) / 30
  df.loc[df.index[-1], 'Evapotranspiration'] = evapotranspiration

  # Ambil Soil Water Reserve sehari sebelumnya
  previous_soil_water_reserve = df['Soil Water Reserve (mm)'].iloc[-1] if len(df) > 1 else 0

  # Hitung Water Balance
  water_balance = previous_soil_water_reserve + current_daily_rainfall - evapotranspiration
  # Simpan Water Balance ke dataframe
  df.loc[df.index[-1], 'Water Balance'] = water_balance

  # Hitung Soil Water Reserve
  df.loc[df.index[-1], 'Soil Water Reserve (mm)'] = min(water_balance, 200)

  # Hitung Water Surplus (WB - 200 >= 0 )
  df.loc[df.index[-1], 'Water Surplus'] = max(0, water_balance - 200)

  # Simpan hasil ke Google Sheets (DB)
  sheet_data.append_row([
      format_datetime(current_time_date),
      estate_name,
      current_daily_rainfall,
      df.loc[df.index[-1], 'Accumulation Rainfall -29 days'],
      df.loc[df.index[-1], 'Evapotranspiration'],
      df.loc[df.index[-1], 'Water Balance'],
      df.loc[df.index[-1], 'Soil Water Reserve (mm)'],
      df.loc[df.index[-1], 'Water Surplus']
  ])

  # Create a new row as a dictionary
  new_row = {
      'Date': pd.to_datetime(format_datetime(current_time_date), dayfirst=True),
      'Estate': estate_name,
      'Daily Rainfall (mm)': current_daily_rainfall,
      'Accumulation Rainfall -29 days': df.loc[df.index[-1], 'Accumulation Rainfall -29 days'],
      'Evapotranspiration': df.loc[df.index[-1], 'Evapotranspiration'],
      'Water Balance': df.loc[df.index[-1], 'Water Balance'],
      'Soil Water Reserve (mm)': df.loc[df.index[-1], 'Soil Water Reserve (mm)'],
      'Water Surplus': df.loc[df.index[-1], 'Water Surplus']
  }

  # Append the new row to the dataframe
  in_df = pd.concat([in_df, pd.DataFrame([new_row])], ignore_index=True)

  return in_df  # Return the modified dataframe

In [35]:
def remove_old_data(df, current_time_date, current_daily_rainfall, estate_name):

  # Remove the last data from dataframe
  filtered_df = df[(df['Estate'] == estate_name) & (df['Date'] == pd.to_datetime(format_datetime(current_time_date), dayfirst=True))]
  if not filtered_df.empty:
    row_index = filtered_df.index[0]
    df.drop(row_index, inplace=True)

  # Remove the last data from spreadsheet
  try:
    cell = sheet_data.find(format_datetime(current_time_date), in_column=1)
    if cell is not None:
      sheet_data.delete_rows(cell.row)
  except gspread.exceptions.CellNotFound:
      print("Data not found in spreadsheet for deletion.")
      sys.exit(1)

  return df

In [36]:
def check_groundwater(accumulation_rainfall, water_surplus):
  if (accumulation_rainfall >= 300) and (water_surplus == 0):
    return True
  elif (accumulation_rainfall >= 60) and (accumulation_rainfall <= 300) and (water_surplus >= 0):
    return True
  else:
    return False

def check_peilscale(peilscale):
  if peilscale <= -51:
    return True
  else:
    return False

def check_season(accumulation_rainfall):
  if accumulation_rainfall < 60 :
    return "Dry"
  elif accumulation_rainfall > 300:
    return "Wet"

def check_rain_in_dry_seasion(daily_rainfall_last_7):
  raining_once = (daily_rainfall_last_7 >= 60).sum() >= 1
  raining_twice = (daily_rainfall_last_7 >= 30).sum() >= 2

  if raining_once or raining_twice:
    return True
  else:
    return False

In [37]:
def validate_water_track(df, current_daily_rainfall, peilscale, next_fertilizer):

  last_row = df.iloc[-1]
  accumulation_rainfall = last_row['Accumulation Rainfall -29 days']
  water_surplus = last_row['Water Surplus']
  daily_rainfall_last_7 = df['Daily Rainfall (mm)'].iloc[-7:]

  # Syarat 1
  validation1 = check_groundwater(accumulation_rainfall, water_surplus)

  # Syarat 2
  print("peilscale", peilscale)
  validation2 = check_peilscale(peilscale)
  print("validation2", validation2)

  # Syarat 3
  season = check_season(accumulation_rainfall)
  validation3 = season not in ["Wet", "Dry"] # if validation3 has value that means it's either 'Wet' or 'Dry', None means it's Optimal

  # Check if season is 'Dry' with rains around 7 days back
  dry_with_rain = False
  if (season == "Dry"):
    dry_with_rain = check_rain_in_dry_seasion(daily_rainfall_last_7)

  return (validation1 and validation2 and validation3), validation1, validation2, season, dry_with_rain

In [38]:
def get_minimal_interval(last_group, next_group):
    return interval_table.get(last_group, {}).get(next_group, 30)  # Default to 30 if not found

In [39]:
def get_alternative_fertilizer(last_group, next_group, last_fertilizer_date, next_fertilizer_date, interval_table, fertilizer_groups):
  recommendation = []
  selisih_hari = (next_fertilizer_date - last_fertilizer_date).days

  for group, fertilizers in fertilizer_groups.items():
      if group != next_group:  # Exclude the desired fertilizer because it hits the interval
          interval = interval_table.get(last_group, {}).get(group, None)
          if interval is not None and selisih_hari >= interval:
              recommendation.extend(fertilizers)

  return recommendation

In [40]:
def get_all_recommendation(last_group, next_group, last_fertilizer_date, next_fertilizer_date, interval_table, fertilizer_groups):
  recommendation = []
  selisih_hari = (next_fertilizer_date - last_fertilizer_date).days

  # Always include the next_group
  if next_group in fertilizer_groups:
      recommendation.extend(fertilizer_groups[next_group])

  # Add other fertilizers that meet the interval
  for group, fertilizers in fertilizer_groups.items():
      if group != next_group:  # Exclude the desired fertilizer
          interval = interval_table.get(last_group, {}).get(group, None)
          if interval is not None and selisih_hari >= interval:
              recommendation.extend(fertilizers)

  return recommendation

In [41]:
def get_fertilizer(groups):
  recommendation = []

  for group, types in groups.items():
    recommendation.append(types)

  return recommendation

In [42]:
def validate_interval_fertilizer(last_group, next_group, last_fertilizer_date, next_fertilizer_date):
  min_interval = get_minimal_interval(last_group, next_group)
  if min_interval == None:  # Handle cases with no defined interval
      return False  # Or return True, depending on how you want to handle these cases
  selisih_hari = (next_fertilizer_date - last_fertilizer_date).days
  return selisih_hari >= min_interval

In [43]:
def get_fertilizer_group(fertilizer):
    for group, types in fertilizer_groups.items():
        if fertilizer in types:
            return group
    return None

In [44]:
def validate_dry_week(fertilizer, df):
  last_days = df['Daily Rainfall (mm)'].iloc[-7:]

  no_rain = 0
  for i in last_days:
    if i == 0:
      no_rain += 1

  return no_rain

In [45]:
def append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, peilscale, last_fertilizer, last_fertilizer_date, next_fertilizer, next_fertilizer_date, reason, recommendation):

  if len(reason) == 0:
    status = "Allowed"
  else:
    status = "Not Allowed"

  # ✅ Simpan Output ke Google Sheets
  output_data = [
      date_input.strftime('%Y-%m-%d %H:%M:%S'),
      username,
      estate_name,
      blok_name,
      current_daily_rainfall,
      peilscale,
      last_fertilizer,
      last_fertilizer_date.strftime("%Y-%m-%d"),
      next_fertilizer,
      next_fertilizer_date.strftime("%Y-%m-%d"),
      status,
      reason,
      recommendation
  ]
  sheet_output.append_row(output_data)

  # ✅Output
  print("\n=== Hasil Analisis Pemupukan ===")
  print(f"Nama User: {username}")
  print(f"Tanggal Input: {date_input.strftime('%Y-%m-%d')}")
  print(f"Curah Hujan: {current_daily_rainfall} mm")
  print(f"Peilscale: {peilscale}")
  print(f"Jenis Pupuk Terakhir: {last_fertilizer} (Tanggal: {last_fertilizer_date.strftime('%Y-%m-%d')})")
  print(f"Plan Jenis Pupuk: {next_fertilizer} (Tanggal: {next_fertilizer_date.strftime('%Y-%m-%d')})")
  print(f"Status: {status}")
  print(f"Reason: {reason}")
  print(f"Rekomendasi: {recommendation}")

  # ✅ System Exit
  raise FertilizerAnalysisComplete

In [46]:
def validate_dolomite(df, last_fertilizer, last_fertilizer_date, next_fertilizer_date, Alternatives):
  dolomite_fertilizer = "Dolomite"

  # Check if the alternatives already has Dolomite inside it
  if dolomite_fertilizer in Alternatives:
    return Alternatives  # Dolomite not allowed

  # 1. Check if the last Daily Rainfall (mm) is < 60
  last_daily_rainfall = df['Daily Rainfall (mm)'].iloc[-1]
  if last_daily_rainfall >= 60:
    return Alternatives  # Dolomite not allowed

  # 2. Check if Accumulation Rainfall is < 300
  accumulation_rainfall = df['Accumulation Rainfall -29 days'].iloc[-1]
  if accumulation_rainfall >= 300:
    return Alternatives  # Dolomite not allowed

  # 3. Check if the interval is met (same as other fertilizers)
  last_group = get_fertilizer_group(last_fertilizer)
  next_group = get_fertilizer_group(dolomite_fertilizer)  # Assuming Dolomite is the next_fertilizer
  min_interval = get_minimal_interval(last_group, next_group)
  selisih_hari = (next_fertilizer_date - last_fertilizer_date).days
  if selisih_hari < min_interval:
    return Alternatives  # Dolomite not allowed

  # If all checks pass, add Dolomite to Alternatives
  Alternatives.append(dolomite_fertilizer)
  return Alternatives

In [47]:
def fill_rainfall_data(df, current_time_date, estate_name):
  # Get missing dates, last reported time, and total missing dates
  missing_dates, last_reported_time, total_missing_dates = get_missing_dates(df, estate_name, current_time_date)

  if total_missing_dates > 0:
    print(border_line)
    print(f"\nTerdapat {total_missing_dates} hari data hujan yang kosong pada estate {estate_name}, antara {format_datetime(current_time_date)} dan {format_datetime(last_reported_time)}")

    for date in missing_dates:
      current_daily_rainfall = float(input(f"Masukkan curah hujan {format_datetime(date.date())} (mm): "))
      df = calculate_rainfall(df, date.date(), current_daily_rainfall, estate_name)

  # Check if today's data is already filled
  is_calculate_rainfall = validate_date(last_reported_time, current_time_date)
  if not is_calculate_rainfall:
    print(border_line)
    current_daily_rainfall = float(input(f"\nMasukkan curah hujan hari ini (mm) untuk estate {estate_name}: "))
    df = calculate_rainfall(df, current_time_date, current_daily_rainfall, estate_name)

  return df

In [48]:
def analyze_fertilizer(df, estate_name, date_input, username, blok_name):

  # Filter the DataFrame for the selected estate
  estate_df = df[df['Estate'] == estate_name]

  # Get the current daily rainfall (last entry for the estate)
  current_daily_rainfall = estate_df['Daily Rainfall (mm)'].iloc[-1]

  #check if today's rainfall is greater than or equal to 60
  reason = ""
  if current_daily_rainfall >= 60:
    reason = "Curah hujan lebih dari 60 mm, pemupukan dihentikan"
    print(reason)
    append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, 0, "", datetime(1970, 1, 1), "", datetime(1970, 1, 1), reason, "")

  #input the next fertilizer date
  next_fertilizer_date = datetime.strptime(input("Masukkan tanggal rencana pupuk (YYYY-MM-DD): "), "%Y-%m-%d")

  #input today's peilscale
  peilscale = float(input("Masukkan nilai Peilscale: "))
  #input the last fertilizer date
  last_fertilizer_date = datetime.strptime(input("Masukkan tanggal pupuk terakhir (YYYY-MM-DD): "), "%Y-%m-%d")
  #input the last fertilizer
  last_fertilizer = input("Masukkan jenis pupuk terakhir: ")
  #input the next fertilizer
  next_fertilizer = input("Masukkan rencana jenis pupuk: ")

  #check the accumulated rainfall data
  validate_water, rain_factor, peilscale_factor, season_factor, dry_with_rain = validate_water_track(df, current_daily_rainfall, peilscale, next_fertilizer)
  if(not validate_water):
    if(not rain_factor):
      reason = "Tidak bisa melakukan pemupukan, karena curah hujan"
      append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, peilscale, "", datetime(1970, 1, 1), "", datetime(1970, 1, 1), reason, "")
    elif(not peilscale_factor):
      reason = "Tidak bisa melakukan pemupukan, karena peilscale di atas -51"
      append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, peilscale, "", datetime(1970, 1, 1), "", datetime(1970, 1, 1), reason, "")
    elif(not season_factor):
      reason = f"Tidak bisa melakukan pemupukan, karena musim {season_factor}"
      if season_factor == "Wet":
          append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, peilscale, "", datetime(1970, 1, 1), "", datetime(1970, 1, 1), reason, "")
      elif season_factor == "Dry":
          print(reason)

  #get the last fertilizer's group
  last_group = get_fertilizer_group(last_fertilizer)
  #get the next fertilizer's group
  next_group = get_fertilizer_group(next_fertilizer)

  #check the interval between the last & the next fertilizer
  validate_interval_result = validate_interval_fertilizer(last_group, next_group, last_fertilizer_date, next_fertilizer_date)

  #check the alternative
  Alternatives = []
  if (not validate_interval_result):
    if last_group == next_group:
      reason = "Karena jarak interval pemupukan di bawah 60 hari"
    elif last_group != next_group:
      reason = "Karena jarak interval pemupukan di bawah 30 hari"
    Alternatives = get_alternative_fertilizer(last_group, next_group, last_fertilizer_date, next_fertilizer_date, interval_table, fertilizer_groups)
  else:
    Alternatives = get_all_recommendation(last_group, next_group, last_fertilizer_date, next_fertilizer_date, interval_table, fertilizer_groups)

  #check the specific fertilizer trait
  #Dolomite
  Alternatives = validate_dolomite(df, last_fertilizer, last_fertilizer_date, next_fertilizer_date, Alternatives)
  #Discard because dry week
  is_dry_week = validate_dry_week(next_fertilizer, df)
  if next_fertilizer == "Urea" and is_dry_week >= 3:
      reason = "3 hari kebelakang tidak terdapat hujan sama sekali"
  elif next_fertilizer in ["Urea", "MOP", "HGFB"] and is_dry_week >= 7:
      reason = "7 hari kebelakang tidak terdapat hujan sama sekali"

  #Join the alternative option
  alternative = ', '.join(Alternatives)
  recommendation = ""
  plan_fertilizer_date = (last_fertilizer_date + timedelta(days=14)).date()
  if (len(Alternatives) != 0):
    recommendation = f"Pupuk alternatif yang disarankan: {alternative}"

  # Append to spreadsheet
  append_to_spreadsheet(date_input, username, estate_name, blok_name, current_daily_rainfall, peilscale, last_fertilizer, last_fertilizer_date, next_fertilizer, next_fertilizer_date, reason, recommendation)

In [49]:
def validate_and_update_last_data(df, estate_name):
  # Filter for the estate's data
  estate_data = df[(df['Estate'] == estate_name)]

  if not estate_data.empty:
    last_date = estate_data['Date'].iloc[-1].date()
    last_rainfall = estate_data['Daily Rainfall (mm)'].iloc[-1]

    print(f"Data terakhir untuk estate {estate_name} pada {format_datetime(last_date)}:")
    print(f"Curah hujan: {last_rainfall} mm")

    is_correct = input("Apakah data ini benar? (y/n): ").lower()

    if is_correct.lower() != "y":
      updated_rainfall = float(input(f"Masukkan curah hujan yang benar untuk {format_datetime(last_date)} (mm): "))

      # Remove the old data
      df = remove_old_data(df, last_date, updated_rainfall, estate_name)

      # Recalculate dependent columns using calculate_rainfall
      df = calculate_rainfall(df, last_date, updated_rainfall, estate_name)

      print(f"Data untuk {format_datetime(last_date)} telah diperbarui.")

  return df

# **User Input**

In [50]:
creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
client = gspread.authorize(creds)
sheet_data = client.open_by_url(sheet_url).worksheet("DB")
sheet_output = client.open_by_url(sheet_url).worksheet("Output")

In [51]:
# Load data dari Google Sheets
data = sheet_data.get_all_records()
df = pd.DataFrame(data)

# Pastikan format kolom benar
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
df['Daily Rainfall (mm)'] = pd.to_numeric(df['Daily Rainfall (mm)'], errors='coerce')

# Cek apakah dataframe kosong
if df.empty:
    print("Dataset kosong! Pastikan ada data di Google Sheets.")

# Drop rows with NaT (Not a Time) values in the 'Date' column
df.dropna(subset=['Date'], inplace=True)

In [52]:
def main_process():
  global df
  current_timezone = pytz.timezone('Asia/Jakarta')
  date_input = datetime.now(current_timezone)
  current_time_date = datetime.now(current_timezone).date()

  username = input("Masukkan Nama Anda: ")

  while True:
    print("\n=== Menu ===")
    print("1. Input Data Hujan")
    print("2. Analisa Pemupukan")
    print("3. Exit")

    choice = input("\nPilih proses (1/2/3): ")

    if choice == "1":

      # Tanyakan kepada pengguna apakah mereka ingin mengganti data hujan terakhir atau menambahkan data hujan yang kosong.
      print("\nApakah Anda ingin mengganti data hujan terakhir (1) atau menambahkan data hujan yang kosong (2)? ")
      sub_choice = input("Pilih proses (1/2):")
      if sub_choice == "1":
        # Validate and update last stored data
        print("\nSebelum input data hujan, silahkan untuk melakukan validasi pada data terakhir")
        estate_name = input("Pilih estate (Inti/Plasma): ")
        df = validate_and_update_last_data(df, estate_name)
      elif sub_choice == "2":
        # Input rain data for both estates from the last data stored in database
        for estate_name in estate:
          df = fill_rainfall_data(df, current_time_date, estate_name)
      else:
          print("Pilihan tidak valid. Kembali ke menu utama.")

    elif choice == "2":

      # Make user fill he missing date first before continue to analyze
      for estate_name in estate:
        df = fill_rainfall_data(df, current_time_date, estate_name)

      # Analyze fertilizer application for a selected estate
      estate_name = input("Pilih estate (Inti/Plasma): ")
      #input estate
      blok_name = input("Masukkan Nama Blok : ")
      if estate_name in estate:
        try:
          analyze_fertilizer(df, estate_name, date_input, username, blok_name)
        except FertilizerAnalysisComplete:
          print("=== Analisis pupuk selesai ===")
      else:
        print("Estate tidak valid. Silakan pilih Inti atau Plasma.")

    elif choice == "3":
      break
    else:
      print("Pilihan tidak valid. Silakan pilih 1, 2, atau 3.")

# Start the main process
# main_process()

# 19-03-2025

# 20-03-2025

In [53]:
# Placeholder for the last date (replace with actual data loading)
def get_last_date_placeholder():
    # return pd.to_datetime("2024-03-20").date() # Example with pandas
    return "2024-03-20"  #  Simplified placeholder

def format_datetime(date_str):
    # In a real app with datetime objects:
    # return date_obj.strftime("%Y-%m-%d")
    return date_str # Simplified for the placeholder

In [5]:
import tkinter as tk
from tkinter import ttk

def submit_action():
    global entry_username, combobox_choice, previous_menu
    if not root_exists:
        return

    username = entry_username.get()
    menu_choice = combobox_choice.get()
    print(f"Username: {username}")
    print(f"Menu Choice: {menu_choice}")

    if menu_choice == "1. Input Data Hujan":
        previous_menu = "main"
        show_rainfall_options()
    elif menu_choice == "2. Analisa Pemupukan":
        previous_menu = "main"
        show_estate_options_for_analysis()

def show_rainfall_options():
    global label_rainfall_option, back_button, current_menu, button_update_rainfall, button_add_rainfall

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "rainfall"
    label_rainfall_option = tk.Label(root, text="Choose Rainfall Option:", font=("Arial", 12))
    label_rainfall_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    # --- Update Rainfall Button ---
    button_update_rainfall = tk.Button(root, text="Update the last Daily Rainfall (mm)", command=goto_update_rainfall, font=("Arial", 10))
    button_update_rainfall.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    # --- Add Rainfall Button ---
    button_add_rainfall = tk.Button(root, text="Add a new Daily Rainfall (mm)", command=goto_add_rainfall, font=("Arial", 10))
    button_add_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")


    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=3, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def goto_update_rainfall():
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Rainfall Option: Update the last Daily Rainfall (mm)")
    previous_menu = "rainfall"
    show_estate_options()

def goto_add_rainfall():
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Rainfall Option: Add a new Daily Rainfall (mm)")
    previous_menu = "rainfall"
    show_estate_options_for_add_rainfall()

def show_estate_options_for_analysis(fertilizer_type):
    # Declare ALL globals at the TOP of the function
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu, \
           entry_blok, entry_tanggal_rencana_pupuk, entry_peilscale, entry_tanggal_pupuk_terakhir, \
           combobox_jenis_pupuk_terakhir, combobox_rencana_jenis_pupuk, label_blok, label_tanggal_rencana_pupuk, \
           label_peilscale, label_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, label_rencana_jenis_pupuk

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "estate_analysis"

    # --- Use sticky="ew" on ALL widgets ---
    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=5, sticky="ew")

    label_blok = tk.Label(root, text="Masukkan Nama Blok:", font=("Arial", 12))
    label_blok.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    entry_blok = tk.Entry(root, font=("Arial", 10))
    entry_blok.grid(row=3, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_rencana_pupuk = tk.Label(root, text="Masukkan tanggal rencana pupuk (YYYY-MM-DD):", font=("Arial", 12))
    label_tanggal_rencana_pupuk.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_rencana_pupuk = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_rencana_pupuk.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    label_peilscale = tk.Label(root, text="Masukkan nilai Peilscale:", font=("Arial", 12))
    label_peilscale.grid(row=6, column=0, padx=10, pady=5, sticky="ew")

    entry_peilscale = tk.Entry(root, font=("Arial", 10))
    entry_peilscale.grid(row=7, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_pupuk_terakhir = tk.Label(root, text="Masukkan tanggal pupuk terakhir (YYYY-MM-DD):", font=("Arial", 12))
    label_tanggal_pupuk_terakhir.grid(row=8, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_pupuk_terakhir = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_pupuk_terakhir.grid(row=9, column=0, padx=10, pady=5, sticky="ew")

    label_jenis_pupuk_terakhir = tk.Label(root, text="Masukkan jenis pupuk terakhir:", font=("Arial", 12))
    label_jenis_pupuk_terakhir.grid(row=10, column=0, padx=10, pady=5, sticky="ew")

    combobox_jenis_pupuk_terakhir = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_jenis_pupuk_terakhir.grid(row=11, column=0, padx=10, pady=5, sticky="ew")

    label_rencana_jenis_pupuk = tk.Label(root, text="Masukkan rencana jenis pupuk:", font=("Arial", 12))
    label_rencana_jenis_pupuk.grid(row=12, column=0, padx=10, pady=5, sticky="ew")

    combobox_rencana_jenis_pupuk = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_rencana_jenis_pupuk.grid(row=13, column=0, padx=10, pady=5, sticky="ew")

    submit_estate_button = tk.Button(root, text="Submit", command=lambda: submit_estate_for_analysis(
        combobox_estate.get(),
        entry_blok.get(),
        entry_tanggal_rencana_pupuk.get(),
        entry_peilscale.get(),
        entry_tanggal_pupuk_terakhir.get(),
        combobox_jenis_pupuk_terakhir.get(),
        combobox_rencana_jenis_pupuk.get()
    ), font=("Arial", 10))
    submit_estate_button.grid(row=14, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=15, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1) # Put this in main

def submit_estate_for_analysis(selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir, jenis_terakhir, rencana_jenis):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Estate: {selected_estate}")
    print(f"Nama Blok: {nama_blok}")
    print(f"Tanggal Rencana Pupuk: {tanggal_rencana}")
    print(f"Nilai Peilscale: {peilscale}")
    print(f"Tanggal Pupuk Terakhir: {tanggal_terakhir}")
    print(f"Jenis Pupuk Terakhir: {jenis_terakhir}")
    print(f"Rencana Jenis Pupuk: {rencana_jenis}")

    previous_menu = "main"
    cancel_to_main()

def submit_rainfall_option(selected_rainfall_option):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Rainfall Option: {selected_rainfall_option}")

    if selected_rainfall_option == "Update the last Daily Rainfall (mm)":
        previous_menu = "rainfall"
        show_estate_options()
    elif selected_rainfall_option == "Add a new Daily Rainfall (mm)":
        # --- Call the new function for adding rainfall ---
        previous_menu = "rainfall"
        show_estate_options_for_add_rainfall()
        # -------------------------------------------------

def show_estate_options_for_add_rainfall():
    global label_estate_option, combobox_estate, submit_estate_add_rainfall_button, back_button, current_menu, entry_daily_rainfall, label_daily_rainfall
    if not root_exists:
        return

    hide_rainfall_widgets()  # Hide rainfall options
    current_menu = "estate_add_rainfall" # Differentiate menu

    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)

    # --- Add Daily Rainfall Input ---
    label_daily_rainfall = tk.Label(root, text="Masukkan Daily Rainfall (mm):", font=("Arial", 12))
    label_daily_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    entry_daily_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_daily_rainfall.grid(row=3, column=0, padx=10, pady=10, sticky="ew")
    # ---------------------------------


    # New button for adding rainfall
    submit_estate_add_rainfall_button = tk.Button(root, text="Submit Estate", command=lambda: submit_estate_for_add_rainfall(combobox_estate.get(), entry_daily_rainfall.get()), font=("Arial", 10))
    submit_estate_add_rainfall_button.grid(row=4, column=0, padx=10, pady=10)


    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=5, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def submit_estate_for_add_rainfall(selected_estate, daily_rainfall):
    global previous_menu
    if not root_exists: return
    print(f"Selected Estate for Add Rainfall: {selected_estate}")
    print(f"Daily Rainfall (mm): {daily_rainfall}")  # Print rainfall
     # Add your logic for handling the new rainfall data here.
    # After processing, you'll likely go back to the main menu:
    previous_menu = "main"
    cancel_to_main()

def show_estate_options():
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu
    if not root_exists:
        return

    hide_rainfall_widgets()
    current_menu = "estate"
    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")
    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)
    submit_estate_button = tk.Button(root, text="Submit Estate", command=lambda: submit_estate(combobox_estate.get()), font=("Arial", 10))
    submit_estate_button.grid(row=2, column=0, padx=10, pady=10)
    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=3, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def submit_estate(selected_estate):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Estate: {selected_estate}")
    previous_menu = "rainfall"
    go_back()

def go_back():
    global previous_menu
    if not root_exists:
        return

    if previous_menu == "main":
        cancel_to_main()
    elif previous_menu == "rainfall":
        hide_estate_widgets()
        hide_rainfall_widgets()
        create_main_widgets()
    elif previous_menu == "estate":
        hide_estate_widgets()
        show_rainfall_options()
    elif previous_menu == "estate_analysis":
        hide_estate_widgets()
        create_main_widgets()
    # Go back from estate_add_rainfall
    elif previous_menu == "estate_add_rainfall":
        hide_estate_widgets()
        show_rainfall_options()

def cancel_to_main():
    if not root_exists:
        return
    hide_rainfall_widgets()
    hide_estate_widgets()
    create_main_widgets()

def hide_main_widgets():
    if not root_exists: return
    try: label_username.grid_forget()
    except AttributeError: pass
    try: entry_username.grid_forget()
    except AttributeError: pass
    try: button_input_hujan.grid_forget()
    except AttributeError: pass
    try: button_analisa_pemupukan.grid_forget()
    except AttributeError: pass
    try: exit_button.grid_forget()
    except AttributeError: pass

def hide_rainfall_widgets():
    if not root_exists: return
    try: label_rainfall_option.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: submit_estate_add_rainfall_button.grid_forget()  # Also hide this one here
    except AttributeError: pass
    # --- Hide the new buttons ---
    try: button_update_rainfall.grid_forget()
    except AttributeError: pass
    try: button_add_rainfall.grid_forget()
    except AttributeError: pass
    # -----------------------------
    if 'current_menu' in globals() :
        global current_menu
        current_menu = None

def hide_estate_widgets():
    if not root_exists: return

    # Use try-except for ALL widget hiding
    try: label_estate_option.grid_forget()
    except AttributeError: pass
    try: combobox_estate.grid_forget()
    except AttributeError: pass
    try: submit_estate_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: label_blok.grid_forget()
    except AttributeError: pass
    try: entry_blok.grid_forget()
    except AttributeError: pass
    try: label_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass
    try: entry_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass
    try: label_peilscale.grid_forget()
    except AttributeError: pass
    try: entry_peilscale.grid_forget()
    except AttributeError: pass
    try: label_tanggal_pupuk_terakhir.grid_forget()
    except AttributeError: pass
    try: entry_tanggal_pupuk_terakhir.grid_forget()
    except AttributeError: pass
    try: label_jenis_pupuk_terakhir.grid_forget()
    except AttributeError: pass
    try: combobox_jenis_pupuk_terakhir.grid_forget()
    except AttributeError: pass
    try: label_rencana_jenis_pupuk.grid_forget()
    except AttributeError: pass
    try: combobox_rencana_jenis_pupuk.grid_forget()
    except AttributeError: pass
    try: submit_estate_add_rainfall_button.grid_forget()
    except AttributeError: pass
    try: entry_daily_rainfall.grid_forget()
    except AttributeError: pass
    try: label_daily_rainfall.grid_forget()
    except AttributeError: pass

    if 'current_menu' in globals():
        global current_menu
        current_menu = None

def create_main_widgets():
    global label_username, entry_username, submit_button, previous_menu, current_menu, back_button, exit_button, button_input_hujan, button_analisa_pemupukan
    if not root_exists:
        return
    root.geometry("500x400") #reset geometry

    current_menu = "main"
    label_username = tk.Label(root, text="Enter Username:", font=("Arial", 12))
    label_username.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    entry_username = tk.Entry(root, font=("Arial", 10))
    entry_username.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    # --- Rainfall Input Button ---
    button_input_hujan = tk.Button(root, text="1. Input Data Hujan", command=goto_input_hujan, font=("Arial", 12))
    button_input_hujan.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    # --- Fertilizer Analysis Button ---
    button_analisa_pemupukan = tk.Button(root, text="2. Analisa Pemupukan", command=goto_analisa_pemupukan, font=("Arial", 12))
    button_analisa_pemupukan.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    exit_button = tk.Button(root, text="Exit", command=on_closing, font=("Arial", 10))
    exit_button.grid(row=5, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

    previous_menu = None
    back_button = None # Back button will be re-created as needed

def goto_input_hujan():
    global previous_menu
    if not root_exists: return
    previous_menu = "main"
    show_rainfall_options()

def goto_analisa_pemupukan():
    global previous_menu, fertilizer_type
    if not root_exists: return
    previous_menu = "main"
    show_estate_options_for_analysis(fertilizer_type)

def disable_buttons():
    """Disables all interactive buttons to prevent further events."""
    global submit_button, back_button, submit_rainfall_button, submit_estate_button, exit_button, submit_estate_add_rainfall_button, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall

    # Use try-except blocks for extra safety.
    try:
        if submit_button: submit_button.config(state="disabled")
    except tk.TclError: pass  # Ignore if widget is already destroyed
    try:
        if back_button: back_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_rainfall_button: submit_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_button: submit_estate_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if exit_button: exit_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_add_rainfall_button: submit_estate_add_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_input_hujan: button_input_hujan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_analisa_pemupukan: button_analisa_pemupukan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_update_rainfall: button_update_rainfall.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_add_rainfall: button_add_rainfall.config(state="disabled")
    except tk.TclError: pass
    
def on_closing():
    global root_exists
    root_exists = False
    disable_buttons()  # Disable buttons immediately
    root.destroy()     # Force immediate destruction of the window

def main_process():
    global root, previous_menu, root_exists, current_menu, \
           submit_button, back_button, submit_rainfall_button, \
           submit_estate_button, exit_button, \
           label_estate_option, combobox_estate, entry_blok, \
           label_tanggal_rencana_pupuk, entry_tanggal_rencana_pupuk, \
           label_peilscale, entry_peilscale, label_tanggal_pupuk_terakhir, \
           entry_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, \
           combobox_jenis_pupuk_terakhir, label_rencana_jenis_pupuk, \
           combobox_rencana_jenis_pupuk, label_rainfall_option, \
           combobox_rainfall, submit_estate_add_rainfall_button, \
           entry_daily_rainfall, label_username, entry_username, \
           label_menu_choice, combobox_choice, label_daily_rainfall, label_blok, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall

    root = tk.Tk()
    root.title("Fertilizer Analysis")
    root.state('zoomed')
    previous_menu = None
    root_exists = True
    current_menu = None

    # Initialize ALL widget variables to None
    label_username = None
    entry_username = None
    exit_button = None
    label_rainfall_option = None
    combobox_rainfall = None
    submit_rainfall_button = None  # No longer directly used
    back_button = None
    label_estate_option = None
    combobox_estate = None
    submit_estate_button = None
    entry_blok = None
    label_tanggal_rencana_pupuk = None
    entry_tanggal_rencana_pupuk = None
    label_peilscale = None
    entry_peilscale = None
    label_tanggal_pupuk_terakhir = None
    entry_tanggal_pupuk_terakhir = None
    label_jenis_pupuk_terakhir = None
    combobox_jenis_pupuk_terakhir = None
    label_rencana_jenis_pupuk = None
    combobox_rencana_jenis_pupuk = None
    submit_estate_add_rainfall_button = None
    entry_daily_rainfall = None
    label_daily_rainfall = None
    label_blok = None
    button_input_hujan = None
    button_analisa_pemupukan = None
    button_update_rainfall = None       # New button
    button_add_rainfall = None        # New button

    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.columnconfigure(0, weight=1)

    create_main_widgets()
    root.mainloop()
    
if __name__ == "__main__":

    fertilizer_type = ["NPK 13", "NPK 15", "NPK 12", "Dolomite", "Urea", "MOP", "HGFB", "CuSO4", "Zincop Chelated", "Kieserite", "RP", "Kaptan", "TSP"]

    main_process()

Selected Rainfall Option: Update the last Daily Rainfall (mm)
Selected Estate: Plasma
Selected Rainfall Option: Update the last Daily Rainfall (mm)
Selected Estate: Plasma
Selected Rainfall Option: Add a new Daily Rainfall (mm)
Selected Estate for Add Rainfall: Inti
Daily Rainfall (mm): 
Selected Rainfall Option: Add a new Daily Rainfall (mm)
Selected Estate for Add Rainfall: Plasma
Daily Rainfall (mm): 
Selected Estate: 
Nama Blok: 
Tanggal Rencana Pupuk: 
Nilai Peilscale: 
Tanggal Pupuk Terakhir: 
Jenis Pupuk Terakhir: 
Rencana Jenis Pupuk: 
Selected Rainfall Option: Update the last Daily Rainfall (mm)
